In [1]:
import numpy as np
import pandas as pd
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df=pd.read_csv("lahore_flats_final.csv")

In [29]:
house=pd.read_csv("lahore_houses_final.csv")

In [30]:
house.columns

Index(['Property ID', 'Society', 'Society Link', 'Name', 'Page Title', 'Price',
       'Area', 'Bedrooms', 'Baths', 'Floor Number', 'Total Floors',
       'Built Year', 'Address', 'Description', 'Features',
       'Nearby Locations and Other Facilities', 'Rooms', 'Other Rooms', 'Link',
       'Built in year', 'Parking Spaces', 'Double Glazed Windows',
       'Central Air Conditioning', 'Central Heating', 'Flooring',
       'Electricity Backup', 'Waste Disposal', 'Floors', 'Other Main Features',
       'Furnished', 'Broadband Internet Access', 'Satellite or Cable TV Ready',
       'Intercom', 'Other Business and Communication Facilities',
       'Community Lawn or Garden', 'Community Swimming Pool', 'Community Gym',
       'First Aid or Medical Centre', 'Day Care Centre', 'Kids Play Area',
       'Barbeque Area', 'Mosque', 'Community Centre',
       'Other Community Facilities', 'Lawn or Garden', 'Swimming Pool',
       'Sauna', 'Jacuzzi', 'Other Healthcare and Recreation Facilities',
 

In [33]:
# ---------------- FLATS ----------------
flats_new = df[[
    'Society',
    'Description',
    'Features'
]].copy()

flats_new['PropertyType'] = 'Flat'


# ---------------- HOUSES ----------------
houses_new = house[[
    'Society',
    'Description',
    'Features'
]].copy()

houses_new['PropertyType'] = 'House'


# ---------------- MERGE ----------------
combined_df = pd.concat(
    [flats_new, houses_new],
    ignore_index=True
)

# preview
combined_df.head()

,Society,Description,Features,PropertyType
0,Askari,12 Marla 4 Bed Apartment: . 4 Bedrooms . 4 Bat...,Built in year : 2026 | Parking Spaces : 2 | Lo...,Flat
1,Askari,This Apartment is located in the Lush green Se...,Built in year : 2025 | Parking Spaces : 1 | Lo...,Flat
2,Askari,ALI REAL ESTATE OFFERS 10 MARLA 3 BEDROOM FLAT...,Built in year : 2024 | Parking Spaces : 4 | Lo...,Flat
3,Askari,RIASAT ESTATE OFFERS ! 4-BEDS APARTMENT AVAILA...,Built in year : 2023 | Parking Spaces : 2 | Lo...,Flat
4,Askari,This Apartment is located in the Lush green Se...,Built in year : 2025 | Lobby in Building | Flo...,Flat


In [34]:
df=combined_df.copy()

In [35]:
df['Features'].value_counts()

Features
Nearby Schools | Nearby Hospitals | Nearby Shopping Malls | Nearby Restaurants | Distance From Airport (kms) | Nearby Public Transport Service | Other Nearby Places | Security Staff | Other Facilities                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     499
Built in year : 2025 | Parking Spaces | Double Glazed Windows | Central Air Conditioning | Central Heating | F

In [36]:
import re

def clean_features(text):

    if pd.isna(text):
        return ""

    text = str(text)

    remove_patterns = [
        r'Built in year\s*:\s*\d+',
        r'Parking Spaces\s*:\s*\d*',
        r'Floor\s*:\s*\d*',
        r'Floors in Building\s*:\s*\d*',
        r'Elevators\s*:\s*\d*',
        r'Distance From Airport \(kms\)',
        r'Other Facilities',
        r'Other Nearby Places',
        r'Other Main Features',
        r'Other Community Facilities',
        r'Other Healthcare and Recreation Facilities',
        r'Other Business and Communication Facilities',
        r'Waste Disposal',
        r'Service Elevators in Building',
        r'Communal/Shared Kitchen',
        r'\bFloor\b',
        r'\bFloors in Building\b',
        r'\bParking Spaces\b'
    ]

    # remove unwanted text
    for pattern in remove_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)

    # remove multiple separators
    text = re.sub(r'(\|\s*)+', '| ', text)

    # remove repeated spaces
    text = re.sub(r'\s+', ' ', text)

    # clean start/end pipes
    text = text.strip(' |')

    return text


# apply
df['clean_features'] = df['Features'].apply(clean_features)

In [37]:
df['clean_features'].value_counts()

clean_features
Double Glazed Windows | Central Air Conditioning | Central Heating | Flooring | Electricity Backup | Floors : 2 | Furnished | Broadband Internet Access | Satellite or Cable TV Ready | Intercom | Community Lawn or Garden | Community Swimming Pool | Community Gym | First Aid or Medical Centre | Day Care Centre | Kids Play Area | Barbeque Area | Mosque | Community Centre | Lawn or Garden | Swimming Pool | Sauna | Jacuzzi | Nearby Schools | Nearby Hospitals | Nearby Shopping Malls | Nearby Restaurants | Nearby Public Transport Service | Maintenance Staff | Security Staff | Facilities for Disabled    4422
Double Glazed Windows | Central Air Conditioning | Central Heating | Flooring | Electricity Backup | Floors : 3 | Furnished | Broadband Internet Access | Satellite or Cable TV Ready | Intercom | Community Lawn or Garden | Community Swimming Pool | Community Gym | First Aid or Medical Centre | Day Care Centre | Kids Play Area | Barbeque Area | Mosque | Community Centre | Lawn

In [38]:
df.shape

(24359, 5)

In [40]:
dict(df['Society'].value_counts())

{'DHA Defence': np.int64(6988),
 'Bahria Town': np.int64(1977),
 'Askari': np.int64(1268),
 'Park View City': np.int64(1263),
 'Raiwind Road': np.int64(1230),
 'Gulberg': np.int64(698),
 'Bahria Orchard': np.int64(660),
 'Johar Town': np.int64(635),
 'Central Park Housing Scheme': np.int64(567),
 'GT Road': np.int64(536),
 'DHA 11 Rahbar': np.int64(492),
 'Valencia Housing Society': np.int64(429),
 'Wapda Town': np.int64(372),
 'Paragon City': np.int64(362),
 'Allama Iqbal Town': np.int64(349),
 'LDA Avenue': np.int64(341),
 'Jubilee Town': np.int64(263),
 'Model Town': np.int64(256),
 'Cantt': np.int64(246),
 'Al Rehman Garden': np.int64(244),
 'Marghzar Officers Colony': np.int64(228),
 'Main Canal Bank Road': np.int64(218),
 'Formanites Housing Scheme': np.int64(215),
 'Defence Road': np.int64(215),
 'Sabzazar Scheme': np.int64(174),
 'Nasheman-e-Iqbal': np.int64(166),
 'Township': np.int64(145),
 'Ferozepur Road': np.int64(145),
 'Eden': np.int64(143),
 'Lake City Meadows': np.int6

In [41]:
society_mapping = {
    
    'Peco Road': 'Pico Road',

}

# apply mapping
df['Society'] = df['Society'].replace(society_mapping)

In [42]:
society_feature_df = df.groupby('Society').agg({
    'clean_features': ' '.join
}).reset_index()


In [43]:
# convert clean_features column into simple python lists

society_feature_df['feature_list'] = society_feature_df['clean_features'].apply(
    lambda x: [i.strip() for i in str(x).split('|') if i.strip()]
)

In [44]:
society_feature_df['feature_list'].value_counts()

feature_list
[Double Glazed Windows, Central Air Conditioning, Central Heating, Flooring, Electricity Backup, Floors : 2, Furnished, Broadband Internet Access, Satellite or Cable TV Ready, Intercom, Community Lawn or Garden, Community Swimming Pool, Community Gym, First Aid or Medical Centre, Day Care Centre, Kids Play Area, Barbeque Area, Mosque, Community Centre, Lawn or Garden, Swimming Pool, Sauna, Jacuzzi, Nearby Schools, Nearby Hospitals, Nearby Shopping Malls, Nearby Restaurants, Nearby Public Transport Service, Maintenance Staff, Security Staff, Facilities for Disabled Double Glazed Windows, Central Air Conditioning, Central Heating, Furnished, Broadband Internet Access, Satellite or Cable TV Ready, Intercom, Community Lawn or Garden, Community Swimming Pool, Community Gym, First Aid or Medical Centre, Day Care Centre, Kids Play Area, Barbeque Area, Mosque, Community Centre, Lawn or Garden, Swimming Pool, Sauna, Jacuzzi, Nearby Schools, Nearby Hospitals, Nearby Shopping Malls, 

In [45]:
society_feature_df['freatureStr']=society_feature_df['feature_list'].apply(''.join)

## vectorization

In [46]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity




In [47]:
# TF-IDF
tfidf_vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2)
)

In [48]:
tfidf_matrix = tfidf_vectorizer.fit_transform(society_feature_df['freatureStr'])

In [49]:
tfidf_matrix.toarray

<bound method _cs_matrix.toarray of <Compressed Sparse Row sparse matrix of dtype 'float64'
	with 28141 stored elements and shape (111, 2435)>>

In [50]:
cosine_sim1 = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [51]:
cosine_sim1.shape

(111, 111)

In [69]:
def recommend_societies(society_name, cosine_sim=cosine_sim1):

    # get index
    idx = society_feature_df[
        society_feature_df['Society'] == society_name
    ].index[0]

    # similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))

    # sort
    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # top 5 similar societies
    sim_scores = sim_scores[1:6]

    # indices
    society_indices = [i[0] for i in sim_scores]

    # dataframe
    recommendations_df = pd.DataFrame({
        'Society': society_feature_df['Society'].iloc[society_indices],
        'SimilarityScore': [i[1] for i in sim_scores]
    })

    return recommendations_df

In [73]:
society_feature_df['Society'].sample(10)

28                                     Eden
68                                    NFC 1
66                               Model Town
102                                Township
21                             College Road
29                              Faisal Town
13     Bankers Co-operative Housing Society
40                              Harbanspura
105              Vital Homes Housing Scheme
81                              Pine Avenue
Name: Society, dtype: object

In [74]:
recommend_societies("Labor Colony")

,Society,SimilarityScore
102,Township,0.667731
105,Vital Homes Housing Scheme,0.665306
53,Lahore - Jaranwala Road,0.665130
109,Walton Road,0.664882
47,Jubilee Town,0.656682


In [56]:
df['Description']

0        12 Marla 4 Bed Apartment: . 4 Bedrooms . 4 Bat...
1        This Apartment is located in the Lush green Se...
2        ALI REAL ESTATE OFFERS 10 MARLA 3 BEDROOM FLAT...
3        RIASAT ESTATE OFFERS ! 4-BEDS APARTMENT AVAILA...
4        This Apartment is located in the Lush green Se...
                               ...                        
24354    20 Marla Elegant And Fully Maintained Super Ho...
24355    1-Kanal Elegant And Fully Maintained Bungalow ...
24356    MIAN BROTHERS OFFERS 100% CONFIRM OPTION IT HA...
24357    ARHAM ESTATE & BUILDERS Offers Proposal House ...
24358    ONE KANAL Bungalow Details: > 5 Master Bed wit...
Name: Description, Length: 24359, dtype: object

In [57]:
df['Description'].value_counts()

Description
* Sarina'S Builder Offers You Ready Houses On Installment Plus Cash * On Installments house Delivered In 4 To 6 Months As Per Your Selected Design on Map location 3 to 5 Years Easy Installments Plan * Payment schedule can adjust as you want it's totally depends on you how to mange it * Model Houses Available For Visit Pictures Are Used As Modal House.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [58]:
df['Description'] = df['Description'].fillna('').astype(str)

society_desc_df = df.groupby('Society').agg({
    'Description': ' '.join
}).reset_index()

In [59]:
tfidf_matrix = tfidf_vectorizer.fit_transform(
    society_desc_df['Description']
)

In [60]:
cosine_sim2 = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [64]:
cosine_sim2.shape

(111, 111)

In [61]:
def recommend_societies(society_name, cosine_sim=cosine_sim2):

    # get index
    idx = society_feature_df[
        society_feature_df['Society'] == society_name
    ].index[0]

    # similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))

    # sort
    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # top 5 similar societies
    sim_scores = sim_scores[1:6]

    # indices
    society_indices = [i[0] for i in sim_scores]

    # dataframe
    recommendations_df = pd.DataFrame({
        'Society': society_feature_df['Society'].iloc[society_indices],
        'SimilarityScore': [i[1] for i in sim_scores]
    })

    return recommendations_df

In [85]:
recommend_societies("Raiwind Road")

,Society,SimilarityScore
25,Defence Road,0.991244
66,Model Town,0.989893
91,Samanabad,0.989333
46,Johar Town,0.987882
10,Bahria Orchard,0.987853


In [86]:
def recommend_societies_with_scores(society_name, top_n=5):

    # combine similarities
    cosine_sim_matrix = (20 * cosine_sim1) + (10 * cosine_sim2)

    # get index
    idx = society_feature_df[
        society_feature_df['Society'] == society_name
    ].index[0]

    # similarity scores
    sim_scores = list(enumerate(cosine_sim_matrix[idx]))

    # sort
    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # top recommendations
    sim_scores = sim_scores[1:top_n+1]

    # indices
    society_indices = [i[0] for i in sim_scores]

    # dataframe
    recommendations_df = pd.DataFrame({
        'Society': society_feature_df['Society'].iloc[society_indices],
        'SimilarityScore': [i[1] for i in sim_scores]
    })

    return recommendations_df

In [87]:
recommend_societies_with_scores('Raiwind Road')

,Society,SimilarityScore
57,Lake City Meadows,23.708987
110,Wapda Town,23.008718
30,Fazaia Housing Scheme,22.891050
60,Main Canal Bank Road,22.833547
28,Eden,22.692007
